# Launcher — test de concurrencia Iceberg

Lanza N workers como procesos independientes, espera a que todos estén listos
y los dispara a la vez con una barrera de fichero.

**worker.py debe estar en el mismo directorio que este notebook.**

In [ ]:
# ── CONFIG DEL TEST ────────────────────────────────────────
# Cada entrada es (operation, target_filter)
# operation: insert | delete | update | read
# target_filter: id de fila a modificar (usado en update/delete)

#WORKERS = [ ("insert", "0") , ("delete", "1") ,]
# worker 0 — inserta filas nuevas
# worker 1 — borra filas con id=1

# Para testear conflicto real (dos workers tocando la misma fila):
WORKERS = [ ("insert", "0"), ("insert", "0"),]   
# worker 0 — update sobre id=42
# worker 1 — update sobre id=42  ← mismo target → conflicto
!pip install pyspark
!sudo apt-get update && sudo apt-get install -y openjdk-17-jdk 
# ──────────────────────────────────────────────────────────

In [ ]:
import subprocess, sys, os, threading, json, glob, time

BARRIER_GO    = "/tmp/all_workers_go"
BARRIER_READY = "/tmp/worker_{}_ready"

# Limpia barreras de runs anteriores
for f in glob.glob("/tmp/worker_*_ready") + [BARRIER_GO]:
    try: os.remove(f)
    except FileNotFoundError: pass

results = []
results_lock = threading.Lock()

def stream(proc, worker_id):
    """Lee stdout del worker línea a línea. JSON → results, texto → LOG."""
    for line in proc.stdout:
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
            with results_lock:
                results.append(record)
            status_icon = {"ok": "✅", "conflict": "⚡", "error": "❌"}.get(record["status"], "?")
            print(
                f"[W{worker_id}] {status_icon} iter={record['iteration']} "
                f"op={record['operation']} "
                f"{record['duration_ms']:.1f}ms"
                + (f" ← {record['error'][:80]}" if record.get('error') else ""),
                flush=True
            )
        except json.JSONDecodeError:
            # Líneas de log de Spark — solo mostrar si no son ruido
            if any(k in line for k in ["INFO:", "WARN", "ERROR", "Exception"]):
                print(f"  [W{worker_id} LOG] {line}", flush=True)

# Entorno limpio — evita que el worker herede la SparkSession del notebook
clean_env = {k: v for k, v in os.environ.items()
             if "SPARK" not in k and "PYSPARK" not in k}

print(f"Lanzando {len(WORKERS)} workers...")
processes = []
threads   = []

for i, (op, target) in enumerate(WORKERS):
    proc = subprocess.Popen(
        [sys.executable, "worker.py", str(i), op, target],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=clean_env,
    )
    t = threading.Thread(target=stream, args=(proc, i), daemon=True)
    t.start()
    processes.append(proc)
    threads.append(t)
    print(f"  Worker {i} arrancado — PID {proc.pid} ({op}, target={target})")

In [ ]:
# ── DISPARAR CUANDO TODOS ESTÉN LISTOS ────────────────────
print(f"Esperando a que {len(WORKERS)} workers inicialicen Spark...")

timeout = 300  # Spark tarda — dale margen
t0 = time.time()
while True:
    ready = glob.glob("/tmp/worker_*_ready")
    print(f"  {len(ready)}/{len(WORKERS)} listos...", end="\r", flush=True)
    if len(ready) >= len(WORKERS):
        break
    if time.time() - t0 > timeout:
        raise TimeoutError("Workers no arrancaron a tiempo")
    time.sleep(0.5)

# Pequeña pausa para que todos lleguen al os.path.exists del bucle
time.sleep(0.1)

# ¡DISPARO!
t_disparo = time.time()
open(BARRIER_GO, "w").close()
print(f"\n🚀 GO — {len(WORKERS)} workers disparados a la vez (t={t_disparo:.6f})")

In [ ]:
# ── ESPERAR A QUE TERMINEN ─────────────────────────────────
for i, proc in enumerate(processes):
    proc.wait()
    print(f"  Worker {i} terminado — exit code {proc.returncode}")

for t in threads:
    t.join(timeout=5)

print(f"\n✅ Todos los workers terminados. {len(results)} resultados recogidos.")

In [ ]:
# ── ANÁLISIS DE RESULTADOS ─────────────────────────────────
import pandas as pd

df = pd.DataFrame(results)

print("=" * 55)
print("RESUMEN POR WORKER")
print("=" * 55)
print(df.groupby(["worker_id", "operation", "status"])["duration_ms"].agg(
    count="count", mean="mean", min="min", max="max"
).round(1).to_string())

print("\n" + "=" * 55)
print("TIEMPO DE REINTENTO POR WORKER")
print("=" * 55)
print(df.groupby("worker_id")["duration_ms"].describe().round(1))

print("\n" + "=" * 55)
print("TIMELINE (orden real de commits)")
print("=" * 55)
df_sorted = df.sort_values("start_time").copy()
t_base = df_sorted["start_time"].min()
df_sorted["t_rel_ms"]            = ((df_sorted["start_time"] - t_base) * 1000).round(1)
df_sorted["gap_desde_anterior"]  = (df_sorted["start_time"].diff() * 1000).round(1)
print(df_sorted[[
    "worker_id", "iteration", "status",
    "duration_ms", "t_rel_ms", "gap_desde_anterior"
]].to_string(index=False))

print("\n" + "=" * 55)
print("CONFLICTOS")
print("=" * 55)
conflicts = df[df["status"] != "ok"]
if conflicts.empty:
    print("  Sin conflictos")
else:
    print(conflicts[[
        "worker_id", "iteration", "status", "duration_ms", "error"
    ]].to_string())

In [ ]:
# ── LIMPIEZA ───────────────────────────────────────────────
import glob, os

for f in glob.glob("/tmp/worker_*_ready") + ["/tmp/all_workers_go"]:
    try:
        os.remove(f)
        print(f"Borrado: {f}")
    except FileNotFoundError:
        pass

print("✅ Barreras limpiadas — listo para el siguiente run")